# 08 Covariance-matching term: pass and fail cases

This notebook isolates `PhysicsComponents.covariance_matching`, the COMET-style term that compares the full interferometric covariance of the predicted profile against the target's covariance, normalised by the target covariance energy.

Modules exercised: `PhysicsComponents.covariance_matching`, `TomoGeometry`.

We engineer matched and mismatched profiles, visualise the covariance error matrices, and trace the term as a scatterer is progressively displaced and broadened.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from configuration.training_config import GeometryConfig
from tools.tomo_geometry            import TomoGeometry
from tools.gaussians                import GaussianMixture
from pipelines.training_pipeline.loss import PhysicsComponents

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-20.0, 40.0, 160).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
xt     = torch.tensor(x_axis, device=DEVICE)
geom   = TomoGeometry(GeometryConfig(), xt)
floor  = 1e-3

def cov_matrix(profile):
    t = to_check_tensor(profile[None, :])
    c = torch.einsum("ijk,bkhw->bijhw", geom.outer, t.to(geom.outer.dtype)) * dx
    return c[0, :, :, 0, 0]

def cov_match(pred, target):
    return float(PhysicsComponents.covariance_matching(to_check_tensor(pred[None, :]), to_check_tensor(target[None, :]), geom.outer, dx, floor))


## Covariance error matrices

For a target scatterer we show the target covariance magnitude, the magnitude for a mean-shifted prediction, and their absolute difference. A mean shift leaves the magnitude structure similar but the term still responds through the complex (phase) entries.

In [ ]:
target  = gaussian_profile(x_axis, 1.0, 8.0, 4.0)
shifted = gaussian_profile(x_axis, 1.0, 16.0, 4.0)

ct = cov_matrix(target)
cs = cov_matrix(shifted)

fig, axes = plt.subplots(1, 3, figsize=(10.0, 3.2))
panels = [(ct.abs(), "|C| target"), (cs.abs(), "|C| mean-shift"), ((cs - ct).abs(), "|C_pred - C_target|")]
for ax, (mat, title) in zip(axes, panels):
    im = ax.imshow(mat.cpu().numpy(), cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("track")
    ax.set_ylabel("track")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

print(f"covariance_matching(target,  target) = {cov_match(target, target):.6e}")
print(f"covariance_matching(shifted, target) = {cov_match(shifted, target):.6e}")

## Term landscape over mean shift and spread

We sweep the predicted scatterer's mean and spread around the target and plot the term as a 2D landscape. The minimum should sit at the true parameters.

In [ ]:
mu_grid  = np.linspace(0.0, 18.0, 31)
sig_grid = np.linspace(1.5, 9.0, 31)
land     = np.empty((sig_grid.size, mu_grid.size))

for i, s in enumerate(sig_grid):
    for j, m in enumerate(mu_grid):
        pred       = gaussian_profile(x_axis, 1.0, float(m), float(s))
        land[i, j] = cov_match(pred, target)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
im = ax.imshow(land, origin="lower", aspect="auto", cmap="magma",
               extent=[mu_grid[0], mu_grid[-1], sig_grid[0], sig_grid[-1]])
ax.scatter([8.0], [4.0], color="cyan", marker="x", s=80, label="true params")
ax.set_xlabel("predicted mean [m]")
ax.set_ylabel("predicted spread [m]")
ax.set_title("covariance_matching landscape")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("term value")
ax.legend()
plt.show()

## Expected outcome

The self-comparison gives a covariance-matching term at the floor. The error matrix is non-trivial for a mean shift, and the 2D landscape has a clear basin whose minimum coincides with the true mean and spread, confirming the term localises the correct scatterer geometry.